# Lesson 19 Lab — KV-Cache Quantization for Long Contexts

**Puzzle:** When context length doubles, why can KV cache dominate even after weight quantization?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Once weights are compressed, KV cache can become the dominant memory term for long contexts and concurrent requests. Quantizing it changes more than capacity: scales must be stored or computed, keys and values are reconstructed inside attention, and small perturbations can change softmax-weighted outputs.


## 0. Predict before running

1. Compute BF16 bytes for K and V with shape `[1,4096,8,128]` before reading the artifact.
2. Predict the ideal INT8 reduction and identify why the measured reduction is smaller than 50%.
3. Choose an output-level metric that is more informative than K/V tensor RMSE alone.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

The KV cache stores keys and values per layer and request. Quantized cache additionally stores scales (and sometimes zero points) at a chosen token/head/block granularity.

- KV bytes scale linearly with batch, layers, sequence, KV heads, head dimension, and two tensors.
- Cache quantization needs scales and often changes attention input error.
- More cache capacity may increase concurrency even when single-request latency does not improve.


## 2. Derive the mechanism

Cache bytes follow `2LBTHD·bytes`, while attention uses `softmax(QKᵀ/√D)V`; quantization error can perturb both logits through `K` and the weighted sum through `V`.

Cache storage is `2·B·S·Hkv·D·bytes`, multiplied by layers in a full model. Quantization adds scale metadata whose granularity may be per tensor, head, token, or block. Attention consumes `softmax(QKᵀ/√D)V`; errors in K affect logits and softmax weights, while errors in V affect the weighted sum. Their consequences are therefore not captured by one raw cache-error number.

Capacity improves only if the backend stores the quantized form persistently rather than dequantizing a full copy. Latency may improve, stay flat, or worsen depending on fused attention support and scale handling.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "19-kv-cache-quantization"
device = require_cuda()
torch.manual_seed(2026 + 19)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | BF16 K and V tensors for one representative long-context attention slice |
| Candidate | INT8 K/V plus explicit scale storage |
| Held constant | batch, sequence 4096, 8 KV heads, head dimension 128, queries, attention computation |
| Measurements | total bytes including scales and attention-output RMSE/cosine |
| Evidence | `pytorch-gpu` |

**Experiment:** Quantize representative KV tensors to INT8 on CUDA, compare bytes and attention-output error, and project capacity across context lengths.


## 5. Read the experiment code

The notebook quantizes real CUDA K/V tensors, includes scale bytes, and compares attention outputs rather than reporting compression alone.

The notebook creates real CUDA K/V tensors, quantizes them, counts code and scale bytes, and evaluates attention outputs against the BF16 reference using the same query tensor. Measuring after the softmax/value path ties numerical error to the consumer of the cache.

This remains a reference implementation. It does not exercise vLLM's FP8 cache format, paged block allocator, per-head scaling, or a fused quantized attention kernel, so service latency is outside the claim.

Only after these variables match the protocol should the cell be executed.


In [2]:
batch,heads,seq,dim=1,8,4096,128; k=torch.randn(batch,heads,seq,dim,device=device); v=torch.randn_like(k); q=torch.randn(batch,heads,1,dim,device=device)
def qdq(t):
    scale=t.abs().amax(-1,keepdim=True).clamp_min(1e-8)/127; qt=torch.round(t/scale).clamp(-128,127).to(torch.int8); return qt,scale,qt.float()*scale
qk,sk,kd=qdq(k); qv,sv,vd=qdq(v)
ref=torch.softmax(q@k.transpose(-1,-2)/(dim**0.5),-1)@v; cand=torch.softmax(q@kd.transpose(-1,-2)/(dim**0.5),-1)@vd
bf16_bytes=2*(k.numel()+v.numel()); int8_bytes=qk.numel()+qv.numel()+sk.numel()*sk.element_size()+sv.numel()*sv.element_size()
result=base_result(19,"pytorch-gpu"); result.update({"shape":{"batch":batch,"heads":heads,"sequence":seq,"head_dim":dim},
    "bf16_bytes":bf16_bytes,"int8_plus_scale_bytes":int8_bytes,"memory_reduction_pct":round((1-int8_bytes/bf16_bytes)*100,4),
    "attention_output_error":error_metrics(ref,cand),"conclusion":"INT8 cache reduced storage in this reference while introducing measurable attention-output error."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| BF16 cache | 16,777,216 bytes |
| INT8 cache plus scales | 8,650,752 bytes |
| Memory reduction | 48.4375% |
| Attention-output RMSE | 0.000231 |
| Attention-output cosine | 0.999958 |


## 7. Interpret rather than merely print

BF16 cache storage was 16,777,216 bytes. INT8 codes plus scales used 8,650,752 bytes, a 48.4375% reduction rather than an ideal 50% because metadata remained. Attention-output RMSE was 0.00023131 with cosine 0.999958 and max absolute error 0.00070267.

The error is small for this random slice, but it is not a language-model quality result. The useful conclusion is that metadata-aware capacity and consumer-level numerical error were both measured; end-to-end quality and fused-kernel cost remain open.

**Inspection rule:** Report cache bytes, metadata, attention error, and any quantize/dequantize overhead separately.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "attention_output_error": {
    "cosine": 0.99995804,
    "mae": 0.00018211,
    "max_abs": 0.00070267,
    "rmse": 0.00023131
  },
  "bf16_bytes": 16777216,
  "conclusion": "INT8 cache reduced storage in this reference while introducing measurable attention-output error.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:58+00:00",
  "int8_plus_scale_bytes": 8650752,
  "lesson": 19,
  "memory_reduction_pct": 48.4375,
  "schema_version": 1,
  "shape": {
    "batch": 1,
    "head_dim": 128,
    "heads": 8,
    "sequence": 4096
  }
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> KV quantization is primarily a capacity decision until end-to-end latency and quality are measured.

**Acceptance/rollback:** Measure actual cache allocation, metadata, context-dependent attention or task error, quant/dequant cost, long-context quality, and end-to-end serving metrics.

**Failure analysis:** Ignoring scale bytes overstates capacity, while comparing cache tensors without attention can understate behavioral impact. A single random context misses layer-dependent and long-range sensitivity. Another failure is to count extra capacity as throughput without testing whether scheduler concurrency and attention latency actually improve.


## 10. Extend the evidence

Repeat by layer/head and context length, compare per-tensor versus per-head scales, and evaluate logit/sequence quality in a small model. Then run a supported vLLM FP8 KV-cache configuration and measure maximum tokens, concurrent requests, TTFT, ITL, and accuracy under the same request set.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
